<a href="https://colab.research.google.com/github/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/Lab6_CineMatch_Labeling_MIK21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 6 · Разметка и подготовка данных

**CineMatch · ДГТУ · МИК21**

Задача: подготовить версионированный набор MovieLens, назначить метки релевантности 0/1/2, защитить временной split от утечек и проверить пилот.

**Инструкция:** [annotation_guideline.md](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/annotation_guideline.md).
**Паспорт:** [Data Card v1.2](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/data_card_update.md).
**Статистика пилота:** [annotation_results.md](https://github.com/localzet/localzet/blob/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/annotation_results.md).

Доступ по публичной ссылке; данные скачиваются при выполнении ячеек (или укажите путь к локальному ZIP). Результаты вычисляются кодом. Второго независимого разметчика-человека нет: применяется альтернативная программная проверка.

In [4]:
from urllib.request import urlretrieve
from pathlib import Path
import sys
BASE = "https://raw.githubusercontent.com/localzet/localzet/main/education/dgtu/ml-creative-industries/lab06-labeling-cinematch/"
for fname in ["prepare_dataset.py", "test_prepare.py", "data_manifest_v1.json", "pilot_labels_v1.csv"]:
    urlretrieve(BASE + fname, fname)
print("Код и зафиксированные артефакты загружены из публичного репозитория")

Код и зафиксированные артефакты загружены из публичного репозитория


In [5]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "-v", "test_prepare"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', '-v', 'test_prepare'], returncode=0)

In [6]:
# Запуск подготовки: выводим причину ошибки, при недоступной сети можно загрузить ZIP вручную.
from pathlib import Path
import subprocess, sys

def run_preparation(source=None):
    cmd = [sys.executable, "prepare_dataset.py", "--output", "prepared"]
    if source is not None:
        cmd.extend(["--source", str(source)])
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr[-9000:])
    return proc

result = run_preparation()
if result.returncode and "DATA_SOURCE_UNAVAILABLE" in (result.stdout + result.stderr):
    print("Автоматическая загрузка недоступна. Загрузите ml-latest-small.zip с GroupLens.")
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Нет доступа к сети. Запустите: python prepare_dataset.py --source путь/к/ml-latest-small.zip")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("ZIP не загружен. Выберите ml-latest-small.zip.")
    result = run_preparation(Path(zip_names[0]))
if result.returncode:
    raise RuntimeError("Подготовка завершилась с ошибкой; подробности напечатаны выше. "
                       "Проверьте доступность источника и контрольные суммы.")
print("Данные подготовлены успешно: prepared/manifest.json")


Источник данных: GitHub (проверенная версия); SHA подтверждён
{
  "version": "cinematch-labels-v1.0.0",
  "observed_raw_ratings": 100836,
  "observed_clean_ratings": 100836,
  "exclusions": 0,
  "split_counts": {
    "train": 62070,
    "validation": 19067,
    "test": 19699
  },
  "class_counts_by_split": {
    "train": {
      "0": 11255,
      "1": 20098,
      "2": 30717
    },
    "validation": {
      "0": 3804,
      "1": 6633,
      "2": 8630
    },
    "test": {
      "0": 4014,
      "1": 6452,
      "2": 9233
    }
  },
  "pilot_items": 12,
  "pilot_automatic_rule_agreement": 12
}

Данные подготовлены успешно: prepared/manifest.json


In [7]:
import json
import pandas as pd
from IPython.display import display
manifest = json.loads(Path("prepared/manifest.json").read_text(encoding="utf-8"))
display(pd.DataFrame(manifest["class_counts_by_split"]).T)
display(pd.DataFrame.from_dict(manifest["split_counts"], orient="index", columns=["records"]))
display(pd.read_csv("prepared/pilot_labels.csv"))
print("IDF обучен только на", manifest["train_unique_movies"], "уникальных фильмах из train")

,0,1,2
train,11255,20098,30717
validation,3804,6633,8630
test,4014,6452,9233


,records
train,62070
validation,19067
test,19699


,pilot_id,source_row,rating,label_v1,control_label,split,agreement
0,R01,9509,4.5,2,2,train,True
1,R02,10414,2.0,0,0,train,True
2,R03,10902,3.5,1,1,train,True
3,R04,35193,5.0,2,2,train,True
4,R05,39161,3.5,1,1,train,True
5,R06,42041,1.0,0,0,train,True
6,R07,62177,5.0,2,2,train,True
7,R08,63697,3.0,1,1,train,True
8,R09,66243,2.5,0,0,train,True
9,R10,89429,4.0,2,2,train,True


IDF обучен только на 7034 уникальных фильмах из train


## Вывод

* Классы **0** (0.5–2.5), **1** (3.0–3.5), **2** (4.0–5.0) назначаются по неизменяемой инструкции. Ненаблюдаемый рейтинг — не 0.
* Связанные события одного пользователя с одинаковым timestamp сохраняются в одном split; предварительно разрешаются повторные `(userId,movieId)`.
* IDF обучается на **train**, изменения test не должны менять веса.
* Программное совпадение двух реализаций не эквивалентно коэффициенту согласия двух людей.
* Файлы prepared содержат исходные псевдонимные ID; **не публикуйте их в открытом репозитории**.

Дальнейшее использование: baseline и сравнение моделей в ЛР 7. Источник: Harper & Konstan (2015), MovieLens Datasets; [GroupLens terms](https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html).